# Building Agentic AutoML

## Goal

This notebook is a hands-on journey to build an Agentic AutoML system from scratch.

Each version introduces one new concept, allowing the agent to evolve step by step while practicing AutoML and agent development.

---

## Version 1

In this version, we build the first version of the agent.

The agent can receive a tabular dataset and a target column, identify whether the task is classification or regression, train a simple LightGBM baseline, evaluate the model, and store the result in a state.

This version keeps the architecture simple.
The agent uses a single model and a simple train-validation split.

## 1. Imports

In [1]:
from pathlib import Path

import pandas as pd

from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split

## 2. Data

We start with a simple tabular dataset.

The target column is provided by the user.

In [2]:
DATA_PATH = "/kaggle/input/datasets/lucalullo/agentic-automl-datasets/simple_binary.csv"
TARGET = "target"

df = pd.read_csv(DATA_PATH)

df.head()

,num_1,num_2,num_3,num_4,category_1,category_2,target
0,-0.144090,-0.172904,-0.111316,0.701984,C,low,0
1,-1.890671,0.690173,0.130746,-0.231793,A,high,0
2,0.246342,1.311081,0.041657,-0.106323,A,high,0
3,0.377065,0.782894,0.585588,-0.580721,B,low,1
4,0.961489,-0.163976,-1.781026,-0.604044,C,medium,0


## 3. Task Detection

The agent first inspects the target to identify the machine learning task.

For now, we use a simple rule based on the number of unique target values.

In [3]:
def detect_task(df, target):
    n_unique = df[target].nunique()

    if n_unique <= 20:
        return "classification"

    return "regression"

In [4]:
task = detect_task(df, TARGET)

task

'classification'

## 4. Feature Detection

The agent identifies numerical and categorical features.

For now, feature types are detected directly from the dataframe dtypes.

In [5]:
def detect_features(df, target):
    X = df.drop(columns=target)

    numerical = X.select_dtypes(include="number").columns.tolist()
    categorical = X.select_dtypes(exclude="number").columns.tolist()

    return numerical, categorical

In [6]:
numerical_features, categorical_features = detect_features(df, TARGET)

numerical_features, categorical_features

(['num_1', 'num_2', 'num_3', 'num_4'], ['category_1', 'category_2'])

## 5. Data Preparation

LightGBM can handle categorical features directly.

The agent converts categorical columns to the pandas `category` dtype and separates features from the target.

In [7]:
def prepare_data(df, target, categorical_features):
    data = df.copy()

    for column in categorical_features:
        data[column] = data[column].astype("category")

    X = data.drop(columns=target)
    y = data[target]

    return X, y

In [8]:
X, y = prepare_data(df, TARGET, categorical_features)

X.dtypes

num_1          float64
num_2          float64
num_3          float64
num_4          float64
category_1    category
category_2    category
dtype: object

## 6. Baseline Model

The agent selects a LightGBM model according to the detected task.

For now, we use the default model configuration.

In [9]:
def select_model(task):
    if task == "classification":
        return LGBMClassifier(random_state=42, verbosity=-1)

    return LGBMRegressor(random_state=42, verbosity=-1)

In [10]:
model = select_model(task)

model

LGBMClassifier(random_state=42, verbosity=-1)

## 7. Metric Selection

The agent selects a simple evaluation metric according to the detected task.

For now, we use ROC AUC for classification and RMSE for regression.

In [11]:
def select_metric(task):
    if task == "classification":
        return "roc_auc"

    return "rmse"

In [12]:
metric = select_metric(task)

metric

'roc_auc'

## 8. Training and Evaluation

The agent splits the dataset into training and validation sets, trains the selected model, and evaluates its performance.

For now, we use a single train-validation split.

In [13]:
def train_and_evaluate(X, y, model, task):
    stratify = y if task == "classification" else None

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=stratify
    )

    model.fit(X_train, y_train)

    if task == "classification":
        predictions = model.predict_proba(X_valid)[:, 1]
        score = roc_auc_score(y_valid, predictions)
    else:
        predictions = model.predict(X_valid)
        score = mean_squared_error(y_valid, predictions) ** 0.5

    return score

In [14]:
score = train_and_evaluate(X, y, model, task)

score

np.float64(0.9068167604752971)

## 9. Agent

We now combine the previous components into a single agent.

The agent receives a dataset and a target column, makes the required decisions, trains the baseline model, and returns its state.

In [15]:
def agent(data_path, target):
    df = pd.read_csv(data_path)

    task = detect_task(df, target)
    numerical, categorical = detect_features(df, target)
    X, y = prepare_data(df, target, categorical)

    model = select_model(task)
    metric = select_metric(task)
    score = train_and_evaluate(X, y, model, task)

    return {
        "task": task,
        "numerical_features": numerical,
        "categorical_features": categorical,
        "model": model.__class__.__name__,
        "metric": metric,
        "score": float(score)
    }

In [16]:
state = agent(DATA_PATH, TARGET)

state

{'task': 'classification',
 'numerical_features': ['num_1', 'num_2', 'num_3', 'num_4'],
 'categorical_features': ['category_1', 'category_2'],
 'model': 'LGBMClassifier',
 'metric': 'roc_auc',
 'score': 0.9068167604752971}

## 10. Regression Test

The same agent should also work with a regression dataset.

We only change the input dataset. The agent must detect the new task and adapt automatically.

In [17]:
REGRESSION_PATH = "/kaggle/input/datasets/lucalullo/agentic-automl-datasets/simple_regression.csv"

regression_state = agent(REGRESSION_PATH, TARGET)

regression_state

{'task': 'regression',
 'numerical_features': ['num_1', 'num_2', 'num_3', 'num_4'],
 'categorical_features': ['category_1', 'category_2'],
 'model': 'LGBMRegressor',
 'metric': 'rmse',
 'score': 1.8760451113860066}

## Notes

The agent can:

- load a tabular dataset
- detect classification or regression
- identify numerical and categorical features
- prepare categorical features for LightGBM
- select a baseline model
- select an evaluation metric
- train and evaluate the model
- return the result through a simple state

The current agent is intentionally limited.

It uses a simple task detection rule, a single train-validation split, one model family, and no preprocessing or optimization strategies.

Future versions will introduce new components and gradually evolve the architecture.